# Item embeddings (Colab, GPU) -- `sentence-t5-base`

Runs the same embedding step as `backend/scripts/build_item_embeddings_local.py`, but on a
Colab GPU instead of your local CPU -- should turn a slow local run into a couple of minutes.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine; A100 if Pro+ gives you one).

**Steps:** run cells top to bottom. Cell 2 mounts Google Drive and reads
`data/processed/item_catalog.parquet` from the project folder. The generated embedding files are
saved directly into that same `data/processed/` folder.

In [13]:
!pip install -q sentence-transformers

In [12]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive", force_remount=True)

# Change this one line if your project has a different Drive folder name.
PROJECT_DIR = Path("/content/drive/Othercomputers/My laptop/generative-recommendation-engine")
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path("/content/drive/Othercomputers/My laptop/SubstrateFinder")
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_DIR}. "
        "Update PROJECT_DIR to the folder containing data/processed/."
    )

os.chdir(PROJECT_DIR)
catalog_path = Path("data/processed/item_catalog.parquet")
if not catalog_path.exists():
    raise FileNotFoundError(f"Catalog not found: {PROJECT_DIR / catalog_path}")

print(f"Working directory: {Path.cwd()}")
print(f"Using catalog: {catalog_path}")
print(os.listdir("data/processed"))

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My laptop/generative-recommendation-engine
Using catalog: data/processed/item_catalog.parquet
['interactions.parquet', 'item_catalog.parquet', 'user_catalog.parquet', 'train.parquet', 'val.parquet', 'test.parquet', 'split_stats.json', 'train_sequences.parquet', 'val_targets.parquet', 'test_targets.parquet', 'interactions.csv', 'item_catalog.csv', 'user_catalog.csv', 'train.csv', 'val.csv', 'test.csv', 'train_sequences.csv', 'val_targets.csv', 'test_targets.csv', 'baseline_results.json', 'baseline_comparison.png', 'item_embeddings.npy', 'item_embeddings_index.parquet', 'semantic_ids.parquet', 'rqvae_params.npz', 'rqvae_metrics.json', 'rqvae_training_curve.png']


In [14]:
import pandas as pd

item_catalog = pd.read_parquet(catalog_path)
item_catalog = item_catalog.sort_values("item_id").reset_index(drop=True)
titles = item_catalog["description"].fillna("").tolist()
print(f"{len(titles):,} item titles")

25,612 item titles


In [4]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [5]:
MODEL_NAME = "sentence-transformers/sentence-t5-base"
model = SentenceTransformer(MODEL_NAME, device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

In [6]:
embeddings = model.encode(
    titles, batch_size=256, show_progress_bar=True, convert_to_numpy=True
)
print("embeddings:", embeddings.shape)

Batches:   0%|          | 0/101 [00:00<?, ?it/s]

embeddings: (25612, 768)


In [20]:
import numpy as np
from sklearn.preprocessing import normalize

embeddings = normalize(embeddings, norm="l2", axis=1).astype("float32")

embedding_path = PROJECT_DIR / "data/processed/item_embeddings.npy"
index_path = PROJECT_DIR / "data/processed/item_embeddings_index.parquet"

np.save(embedding_path, embeddings)
item_catalog[["item_id"]].to_parquet(index_path)
print("saved embeddings:", embedding_path)
print("saved index:", index_path)
print("embeddings shape:", embeddings.shape)

saved embeddings: /content/drive/Othercomputers/My laptop/generative-recommendation-engine/data/processed/item_embeddings.npy
saved index: /content/drive/Othercomputers/My laptop/generative-recommendation-engine/data/processed/item_embeddings_index.parquet
embeddings shape: (25612, 768)


In [21]:
from pathlib import Path
import numpy as np
import pandas as pd

embedding_path = Path("data/processed/item_embeddings.npy")
index_path = Path("data/processed/item_embeddings_index.parquet")

print("Working directory:", Path.cwd())
print("Working directory is on Drive:", str(Path.cwd()).startswith("/content/drive/"))
print("Embeddings file:", embedding_path.resolve())
print("Index file:", index_path.resolve())
print("Embeddings exists:", embedding_path.exists())
print("Index exists:", index_path.exists())

saved_embeddings = np.load(embedding_path, mmap_mode="r")
saved_index = pd.read_parquet(index_path)
print("Embeddings shape:", saved_embeddings.shape)
print("Index rows:", len(saved_index))
print("Files match:", saved_embeddings.shape[0] == len(saved_index))
print("Embeddings size (MB):", round(embedding_path.stat().st_size / 1024**2, 2))
print("Index size (MB):", round(index_path.stat().st_size / 1024**2, 2))

Working directory: /content/drive/Othercomputers/My laptop/generative-recommendation-engine
Working directory is on Drive: True
Embeddings file: /content/drive/Othercomputers/My laptop/generative-recommendation-engine/data/processed/item_embeddings.npy
Index file: /content/drive/Othercomputers/My laptop/generative-recommendation-engine/data/processed/item_embeddings_index.parquet
Embeddings exists: True
Index exists: True
Embeddings shape: (25612, 768)
Index rows: 25612
Files match: True
Embeddings size (MB): 75.04
Index size (MB): 0.15


In [8]:
# same sanity check as the local script -- do nearest neighbors look related?
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

rng = np.random.RandomState(0)
for idx in rng.choice(len(embeddings), 3, replace=False):
    sims = cosine_similarity(embeddings[idx:idx+1], embeddings)[0]
    top = np.argsort(-sims)[1:6]
    print(f"\n'{titles[idx]}' nearest neighbors:")
    for j in top:
        print(f"  {sims[j]:.3f}  {titles[j]}")


'Jeecoo V20 Stereo Gaming Headset for PS4 PS5 Xbox One - Over Ear Headphones with Noise Cancelling Microphone - LED Light Soft Earmuffs for PC Laptops Mobiles' nearest neighbors:
  0.962  Jeecoo J20 Stereo Gaming Headset for PS4, Pro, Xbox One S, Xbox One Controller, Noise Cancelling Over Ear Headphones with Mic, Bass Surround Soft Memory Earmuffs for PC Nintendo Switch Games
  0.959  PS4 Gaming Headset | Xbox One Headset |Xbox One S Headset with Microphone VOTRON Over Ear Stereo Gaming Headphones with LED Light Noise Reduction for Xbox One PS4 PC Mac iPad PSP Headphones
  0.959  Kootop Stereo Gaming Headset for Xbox one ,PS4 PC, Noise Cancelling Over Ear Headphones with Mic,Soft Earmuffs ,Bass Surround ,LED Light ,for Laptop Tablet Phone(Black&Blue)
  0.958  Jeecoo J20 Gaming Headset for PS4 New Xbox One, Stereo Over-Ear Headphones with Mic for PC Computer Mac Laptop Nintendo Switch Games
  0.958  BENGOO V-4 Gaming Headset for Xbox One, PS4, PC, Controller, Noise Cancelling Over Ear 

## Results

The files are saved directly to the synced Drive folder:

- `data/processed/item_embeddings.npy`
- `data/processed/item_embeddings_index.parquet`

You can also download local copies in the next cell.

In [ ]:
from google.colab import files

files.download(str(embedding_path))
files.download(str(index_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>